# Local Feature Sampling — Large Maze Ellipsoids Diffuser

Trains a VE-SDE score model on `custom/largemaze_ellipsoids-v1` with **local feature
sampling** for environment conditioning.  Occupancy map features are bilinearly sampled
at each noisy waypoint's position rather than being globally pooled into a FiLM vector.

Preview samples are drawn from the held-out `custom/largemaze_ellipsoids_2-v1` dataset.

In [ ]:
# ============================================================
# 1. Hyperparameters
# ============================================================

_DATA_ROOT = "../Diffuser/TrajectoryDatasetGeneration/data/custom"

TRAIN_RASTERIZED_DIR = f"{_DATA_ROOT}/largemaze_ellipsoids-v1_rasterized"
EVAL_RASTERIZED_DIR  = f"{_DATA_ROOT}/largemaze_ellipsoids_2-v1_rasterized"

MAP_H = 64
MAP_W = 64

UNET_INPUT_DIM = 32
DIM_MULTS      = (1, 2, 4)
LOCAL_DIM      = 16

SIGMA_MIN = 0.01
SIGMA_MAX = 10.0
N_LEVELS  = 1000

P_UNCOND       = 0.1
GUIDANCE_SCALE = 0.6

BATCH_SIZE       = 128
LR               = 3e-4
TOTAL_STEPS      = 1_000_000
EMA_DECAY        = 0.9999
EMA_START_STEP   = 5_000
EMA_UPDATE_EVERY = 10

N_SAMPLE_STEPS = 25

LOG_EVERY     = 5000
PREVIEW_EVERY = 5000
SAVE_EVERY    = 50_000

## 2. Imports

In [ ]:
import sys, os, json, copy, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
for p in [PROJECT_ROOT, NOTEBOOK_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from EllipsoidalCBFSampling.models.score_net_ellipsoids_rasterizedlocalfeats import TemporalUnet
from EllipsoidalCBFSampling.models.ve_diffusion_ellipsoids_rasterizedlocalfeats import VEDiffusion
from EllipsoidalCBFSampling.models.samplers_ellipsoids_cfg_rasterizedlocalfeats import dpm_solver_1_cfg_sample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

## 3. Dataset

In [ ]:
class RasterizedEllipsoidsDataset(Dataset):
    """
    Loads a pre-rasterized dataset produced by convert_to_rasterized.py.

    Returns per item:
        traj    : [T_steps, 2]   float32 — x,y waypoints, normalised to [-1,1]
        occ_map : [1, H, W]      float32 — binary occupancy (walls + ellipsoids)

    ellipsoids_world is kept as a numpy array for visualisation only.
    """

    def __init__(self, rasterized_dir, T_steps=None):
        rasterized_dir = os.path.abspath(rasterized_dir)
        with open(os.path.join(rasterized_dir, 'metadata.json')) as f:
            meta = json.load(f)

        self.xy_min   = np.array(meta['xy_min'],   dtype=np.float32)
        self.xy_range = np.array(meta['xy_range'], dtype=np.float32)
        self.xy_max   = self.xy_min + self.xy_range
        ep_dir        = os.path.join(rasterized_dir, 'episodes')

        trajs, occ_maps, ell_world = [], [], []

        for i in range(meta['n_episodes']):
            ep   = np.load(os.path.join(ep_dir, f'episode_{i}.npz'))
            traj = ep['trajectory_norm'][:, :2].astype(np.float32)   # [T, 2]

            if T_steps is not None:
                if traj.shape[0] >= T_steps:
                    traj = traj[:T_steps]
                else:
                    pad  = np.repeat(traj[-1:], T_steps - traj.shape[0], axis=0)
                    traj = np.concatenate([traj, pad], axis=0)

            trajs.append(traj)
            occ_maps.append(ep['occ_map'].astype(np.float32))
            ell_world.append(ep['ellipsoids_world'].astype(np.float32))

        self.horizon          = T_steps if T_steps is not None else trajs[0].shape[0]
        self.trajs            = torch.tensor(np.stack(trajs),    dtype=torch.float32)
        self.occ_maps         = torch.tensor(np.stack(occ_maps), dtype=torch.float32)
        self.ellipsoids_world = np.stack(ell_world)   # numpy, for viz only

        print(f'[{os.path.basename(rasterized_dir)}] {len(self)} eps, horizon={self.horizon}')
        print(f'  xy_min={self.xy_min}  xy_max={self.xy_max}')

    def __len__(self):  return len(self.trajs)
    def __getitem__(self, idx):  return self.trajs[idx], self.occ_maps[idx]

    def unnormalize(self, x):
        xy_min   = torch.tensor(self.xy_min,   dtype=x.dtype, device=x.device)
        xy_range = torch.tensor(self.xy_range, dtype=x.dtype, device=x.device)
        return (x + 1.0) / 2.0 * xy_range + xy_min

    def normalize(self, x):
        xy_min   = torch.tensor(self.xy_min,   dtype=x.dtype, device=x.device)
        xy_range = torch.tensor(self.xy_range, dtype=x.dtype, device=x.device)
        return 2.0 * (x - xy_min) / xy_range - 1.0


# Load training dataset
ds      = RasterizedEllipsoidsDataset(TRAIN_RASTERIZED_DIR)
dl      = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
T_STEPS = ds.horizon

# Derive actual bitmap size from loaded data
MAP_H, MAP_W = ds.occ_maps.shape[-2], ds.occ_maps.shape[-1]
print(f'occ_map size: {MAP_H}x{MAP_W}')

# Load eval dataset (held-out; used for preview sampling)
ds_eval = RasterizedEllipsoidsDataset(EVAL_RASTERIZED_DIR, T_steps=T_STEPS)

## 4. Visualisation helpers & bitmap preview

In [ ]:
_MAZE_MAP = [
    [1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,1,0,0,0,0,0,1],
    [1,0,1,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,1,0,1,1,1,0,1],
    [1,0,0,1,0,1,0,0,0,0,0,1],
    [1,1,0,1,0,1,0,1,0,1,1,1],
    [1,0,0,1,0,0,0,1,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1],
]

def draw_maze(ax):
    for i, row in enumerate(_MAZE_MAP):
        for j, cell in enumerate(row):
            if cell:
                rect = plt.Rectangle((-5.5+j-0.5, 4.0-i-0.5), 1.0, 1.0,
                                     linewidth=0, facecolor='dimgray')
                ax.add_patch(rect)
    ax.set_xlim(-6, 6); ax.set_ylim(-4.5, 4.5)

def draw_ellipsoid(ax, cx, cy, a, b, color='steelblue', alpha=0.4):
    t = np.linspace(0, 2*np.pi, 200)
    x = cx + a * np.sign(np.cos(t)) * np.sqrt(np.abs(np.cos(t)))
    y = cy + b * np.sign(np.sin(t)) * np.sqrt(np.abs(np.sin(t)))
    ax.add_patch(plt.Polygon(np.column_stack([x, y]),
                              facecolor=color, edgecolor=color, alpha=alpha))

# Preview: world-space view alongside pre-loaded rasterized bitmap
sample_traj, sample_occ = ds[0]
sample_ell = ds.ellipsoids_world[0]   # [n_ell, 4] for drawing

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
draw_maze(ax1)
for row in sample_ell:
    cx, cy, a, b = row
    if a > 1e-4: draw_ellipsoid(ax1, cx, cy, a, b)
traj_world = ds.unnormalize(sample_traj)
ax1.plot(traj_world[:,0], traj_world[:,1], color='royalblue', lw=1.5)
ax1.set_title('World coords'); ax1.set_aspect('equal')
ax2.imshow(sample_occ[0].numpy(), origin='lower', cmap='gray_r')
ax2.set_title(f'Pre-rasterized occupancy {MAP_H}x{MAP_W}')
plt.tight_layout(); plt.show()

## 5. Model Setup

In [ ]:
score_net = TemporalUnet(
    state_dim=2, T_steps=T_STEPS,
    unet_input_dim=UNET_INPUT_DIM, dim_mults=DIM_MULTS,
    local_dim=LOCAL_DIM,
).to(device)

ve = VEDiffusion(model=score_net, sigma_min=SIGMA_MIN, sigma_max=SIGMA_MAX, n_levels=N_LEVELS).to(device)

ema_model = copy.deepcopy(score_net).to(device)
for p in ema_model.parameters(): p.requires_grad_(False)

optimizer = torch.optim.Adam(score_net.parameters(), lr=LR)

n_params = sum(p.numel() for p in score_net.parameters())
print(f'TemporalUnet params: {n_params:,}')

# Sanity check
_x  = torch.randn(4, T_STEPS, 2, device=device)
_s  = torch.ones(4, device=device)
_xs = _x[:, 0]
_xg = _x[:, -1]
_om = torch.zeros(4, 1, MAP_H, MAP_W, device=device)
print('forward shape:', score_net(_x, _s, _xs, _xg, _om).shape)

In [ ]:
CKPT_DIR  = os.path.join(NOTEBOOK_DIR, 'checkpoints', 'rasterized_localfeats')
CKPT_NAME = None   # set to a filename to load an existing checkpoint
os.makedirs(CKPT_DIR, exist_ok=True)

## 6. Training + Live Preview

In [ ]:
import itertools

# Fixed eval samples — renormalized from eval space to training space
rng_fix  = np.random.default_rng(42)
fix_idx  = rng_fix.choice(len(ds_eval), size=4, replace=False).tolist()

fix_trajs_eval, fix_occ_eval = zip(*[ds_eval[i] for i in fix_idx])
fix_trajs_eval  = torch.stack(fix_trajs_eval)           # [4, T, 2] eval-norm space
fix_occ         = torch.stack(fix_occ_eval).to(device)  # [4, 1, H, W] pre-loaded
fix_ells_world  = ds_eval.ellipsoids_world[fix_idx]     # [4, n_ell, 4] numpy for viz

# Renormalize trajectories from eval space to training space
fix_trajs_world = ds_eval.unnormalize(fix_trajs_eval)
fix_trajs       = ds.normalize(fix_trajs_world)
fix_xs = fix_trajs[:, 0, :].to(device)
fix_xg = fix_trajs[:, -1, :].to(device)

def preview(step):
    ema_model.eval()
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Step {step:,}  CFG w={GUIDANCE_SCALE}  [eval set]', fontsize=11)
    with torch.no_grad():
        for j, ax in enumerate(axes):
            samp = dpm_solver_1_cfg_sample(
                ema_model, ve,
                x_start=fix_xs[j:j+1], x_goal=fix_xg[j:j+1],
                occ_map=fix_occ[j:j+1],
                T_steps=T_STEPS, n_steps=N_SAMPLE_STEPS,
                guidance_scale=GUIDANCE_SCALE, device=device,
            )
            world = ds.unnormalize(samp.cpu())
            xs_w  = ds.unnormalize(fix_xs[j:j+1].cpu())
            xg_w  = ds.unnormalize(fix_xg[j:j+1].cpu())
            draw_maze(ax)
            for row in fix_ells_world[j]:
                cx, cy, a, b = row
                if a > 1e-4: draw_ellipsoid(ax, cx, cy, a, b)
            ax.plot(world[0,:,0], world[0,:,1], lw=1.2, color='steelblue', alpha=0.85)
            ax.scatter(xs_w[0,0], xs_w[0,1], c='green', s=60, zorder=5)
            ax.scatter(xg_w[0,0], xg_w[0,1], c='red',   s=60, zorder=5)
            ax.set_aspect('equal')
    plt.tight_layout(); plt.show()
    ema_model.train()

# ---- Training loop ----
loader_iter   = itertools.cycle(dl)
loss_history  = []
recent_losses = []
start_step    = 0

score_net.train()
t0 = time.time()

for step in range(start_step + 1, TOTAL_STEPS + 1):
    traj, occ_map = next(loader_iter)
    traj    = traj.to(device)
    occ_map = occ_map.to(device)

    xs = traj[:, 0, :]
    xg = traj[:, -1, :]

    loss, info = ve(traj, xs, xg, occ_map, p_uncond=P_UNCOND)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step >= EMA_START_STEP and step % EMA_UPDATE_EVERY == 0:
        with torch.no_grad():
            for p_ema, p in zip(ema_model.parameters(), score_net.parameters()):
                p_ema.data.mul_(EMA_DECAY).add_(p.data, alpha=1.0 - EMA_DECAY)

    loss_history.append(info['loss'])
    recent_losses.append(info['loss'])

    if step % LOG_EVERY == 0:
        avg = float(np.mean(recent_losses)); recent_losses.clear()
        elapsed = time.time() - t0
        sps = (step - start_step) / elapsed if elapsed > 0 else 0
        print(f'step {step:>7,}  loss={avg:.4f}  sigma={info["sigma_mean"]:.3f}  {sps:.1f} sps')

    if step % PREVIEW_EVERY == 0:
        preview(step)

    if step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f've_unet_largemaze_localfeats_step{step}.pt')
        torch.save({
            'step': step, 'score_net': score_net.state_dict(),
            'ema_model': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
            'loss_history': loss_history,
            'config': {
                'T_steps': T_STEPS, 'unet_input_dim': UNET_INPUT_DIM,
                'dim_mults': list(DIM_MULTS), 'local_dim': LOCAL_DIM,
                'sigma_min': SIGMA_MIN, 'sigma_max': SIGMA_MAX, 'n_levels': N_LEVELS,
                'map_h': MAP_H, 'map_w': MAP_W,
                'p_uncond': P_UNCOND, 'guidance_scale': GUIDANCE_SCALE,
                'train_rasterized_dir': TRAIN_RASTERIZED_DIR,
                'eval_rasterized_dir':  EVAL_RASTERIZED_DIR,
                'xy_min': ds.xy_min.tolist(), 'xy_max': ds.xy_max.tolist(),
            },
        }, ckpt_path)
        print(f'  checkpoint saved: step {step:,}')

print('Done.')

## 7. Checkpoint — Save / Load

In [ ]:
# Manual save
CKPT_SAVE_PATH = os.path.join(CKPT_DIR, f've_unet_largemaze_localfeats_step{step}.pt')
torch.save({
    'step': step, 'score_net': score_net.state_dict(),
    'ema_model': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
    'loss_history': loss_history,
    'config': {
        'T_steps': T_STEPS, 'unet_input_dim': UNET_INPUT_DIM,
        'dim_mults': list(DIM_MULTS), 'local_dim': LOCAL_DIM,
        'sigma_min': SIGMA_MIN, 'sigma_max': SIGMA_MAX, 'n_levels': N_LEVELS,
        'map_h': MAP_H, 'map_w': MAP_W,
        'p_uncond': P_UNCOND, 'guidance_scale': GUIDANCE_SCALE,
        'train_rasterized_dir': TRAIN_RASTERIZED_DIR,
        'eval_rasterized_dir':  EVAL_RASTERIZED_DIR,
        'xy_min': ds.xy_min.tolist(), 'xy_max': ds.xy_max.tolist(),
    },
}, CKPT_SAVE_PATH)
print(f'Saved: {CKPT_SAVE_PATH}')

In [ ]:
# Load checkpoint
LOAD_PATH = os.path.join(CKPT_DIR, 've_unet_largemaze_localfeats_step1000000.pt')
ckpt = torch.load(LOAD_PATH, map_location=device, weights_only=False)
cfg  = ckpt['config']

T_STEPS        = cfg['T_steps']
UNET_INPUT_DIM = cfg['unet_input_dim']
DIM_MULTS      = tuple(cfg['dim_mults'])
LOCAL_DIM      = cfg['local_dim']
MAP_H          = cfg.get('map_h', 64)
MAP_W          = cfg.get('map_w', 64)
GUIDANCE_SCALE = cfg.get('guidance_scale', 0.6)

score_net = TemporalUnet(
    state_dim=2, T_steps=T_STEPS,
    unet_input_dim=UNET_INPUT_DIM, dim_mults=DIM_MULTS,
    local_dim=LOCAL_DIM,
).to(device)
ve = VEDiffusion(
    model=score_net,
    sigma_min=cfg['sigma_min'], sigma_max=cfg['sigma_max'], n_levels=cfg['n_levels'],
).to(device)
ema_model = copy.deepcopy(score_net).to(device)
for p in ema_model.parameters(): p.requires_grad_(False)

score_net.load_state_dict(ckpt['score_net'])
ema_model.load_state_dict(ckpt['ema_model'])

# Reload datasets from the rasterized dirs stored in the checkpoint
_train_dir = cfg.get('train_rasterized_dir', TRAIN_RASTERIZED_DIR)
_eval_dir  = cfg.get('eval_rasterized_dir',  EVAL_RASTERIZED_DIR)
ds      = RasterizedEllipsoidsDataset(_train_dir)
ds_eval = RasterizedEllipsoidsDataset(_eval_dir, T_steps=T_STEPS)

print(f'Loaded checkpoint from step {ckpt["step"]:,}')

## 8. Final Sampling — Grid

In [ ]:
# Sample from eval set and visualise
ema_model.eval()

rng = np.random.default_rng(0)
test_idx = rng.choice(len(ds_eval), size=8, replace=False).tolist()

test_trajs_eval, test_occ_eval = zip(*[ds_eval[i] for i in test_idx])
test_trajs_eval = torch.stack(test_trajs_eval)
test_occ        = torch.stack(test_occ_eval).to(device)   # [8, 1, H, W] pre-loaded
test_ells_world = ds_eval.ellipsoids_world[test_idx]      # [8, n_ell, 4] for viz

# Renormalize trajectories from eval space to training space
test_trajs_world = ds_eval.unnormalize(test_trajs_eval)
test_trajs       = ds.normalize(test_trajs_world)
test_xs = test_trajs[:, 0, :].to(device)
test_xg = test_trajs[:, -1, :].to(device)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
fig.suptitle(f'DPM-Solver-1 CFG  w={GUIDANCE_SCALE}  {N_SAMPLE_STEPS} steps  [eval set]', fontsize=12)

with torch.no_grad():
    for j, ax in enumerate(axes):
        samp = dpm_solver_1_cfg_sample(
            ema_model, ve,
            x_start=test_xs[j:j+1], x_goal=test_xg[j:j+1],
            occ_map=test_occ[j:j+1],
            T_steps=T_STEPS, n_steps=N_SAMPLE_STEPS,
            guidance_scale=GUIDANCE_SCALE, device=device,
        )
        world = ds.unnormalize(samp.cpu())
        xs_w  = ds.unnormalize(test_xs[j:j+1].cpu())
        xg_w  = ds.unnormalize(test_xg[j:j+1].cpu())

        draw_maze(ax)
        for row in test_ells_world[j]:
            cx, cy, a, b = row
            if a > 1e-4: draw_ellipsoid(ax, cx, cy, a, b)
        ax.plot(world[0,:,0], world[0,:,1], lw=1.5, color='steelblue')
        ax.scatter(xs_w[0,0], xs_w[0,1], c='green', s=80, zorder=5)
        ax.scatter(xg_w[0,0], xg_w[0,1], c='red',   s=80, zorder=5)
        ax.set_aspect('equal')

plt.tight_layout(); plt.show()

## 9. Loss Curve

In [ ]:
if 'loss_history' in ckpt:
    lh = np.array(ckpt['loss_history'])
else:
    lh = np.array(loss_history)

w = 200
ma = np.convolve(lh, np.ones(w)/w, mode='valid')
steps = np.arange(len(lh))

plt.figure(figsize=(12, 4))
plt.plot(steps, lh, alpha=0.2, color='steelblue', label='raw')
plt.plot(steps[w-1:], ma, color='steelblue', label=f'{w}-step MA')
plt.xlabel('Step'); plt.ylabel('DSM Loss')
plt.title('Training Loss — Local Feature Sampling Model')
plt.legend(); plt.tight_layout(); plt.show()